# Module 9 • Machine Translation

# Lesson 55 • Machine Translation Evaluation — BLEU, chrF, COMET, Semantic Similarity, and Statistical Significance

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU only for the core lesson

---

## Scope

A strong machine-translation experiment requires more than one automatic score.

This lesson covers:

- BLEU;
- chrF and chrF++;
- COMET;
- semantic embedding similarity;
- sentence-level versus corpus-level evaluation;
- mean and standard deviation;
- 95% confidence intervals;
- bootstrap resampling;
- paired bootstrap system comparison;
- approximate randomization;
- effect size;
- metric correlation;
- Arabic and tashkeel-sensitive evaluation;
- reproducible reporting.

The notebook is fully executable offline. Neural metrics such as COMET and SBERT are
included as optional templates because they require external model downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- compute corpus BLEU from first principles;
- explain modified n-gram precision and brevity penalty;
- compute a chrF-style character metric;
- distinguish sentence-level and corpus-level metrics;
- calculate mean, standard deviation, and 95% confidence intervals;
- bootstrap uncertainty around system scores;
- perform paired system comparisons;
- interpret p-values and effect sizes;
- explain COMET and embedding-based semantic evaluation;
- design Arabic-specific evaluation that does not hide tashkeel errors;
- produce a rigorous MT evaluation table.

## Table of Contents

1. Why MT Evaluation Needs Multiple Metrics  
2. Reference-Based Evaluation  
3. Corpus-Level vs Sentence-Level Scores  
4. BLEU  
5. Modified n-Gram Precision  
6. Brevity Penalty  
7. Corpus BLEU Implementation  
8. BLEU Limitations  
9. chrF  
10. chrF++  
11. chrF Implementation  
12. Semantic Similarity  
13. Offline Semantic Proxy  
14. COMET  
15. Optional COMET Workflow  
16. Optional SentenceTransformer Workflow  
17. Evaluation Corpus  
18. Competing MT Systems  
19. Corpus-Level Metrics  
20. Sentence-Level Metrics  
21. Mean  
22. Standard Deviation  
23. Standard Error  
24. 95% Confidence Interval  
25. Bootstrap Confidence Interval  
26. Why Bootstrap MT Metrics  
27. Paired Bootstrap Comparison  
28. Approximate Randomization  
29. Statistical Significance  
30. Practical Significance  
31. Effect Size  
32. Metric Correlation  
33. System Ranking Stability  
34. Error Categories  
35. Per-Category Evaluation  
36. Arabic-Specific Evaluation  
37. Tashkeel Preservation  
38. Tashkeel-Sensitive Exact Match  
39. Vocalized vs Unvocalized Evaluation  
40. Reporting BLEU Correctly  
41. Reporting chrF Correctly  
42. Reporting COMET Correctly  
43. Reproducible Evaluation Table  
44. Confidence Interval Visualization  
45. Significance Matrix  
46. Interpretation  
47. Common Evaluation Mistakes  
48. Recommended MT Evaluation Protocol  
49. Reproducibility  
50. Knowledge Check  
51. Exercises  
52. Summary and Next Lesson

# 1. Why MT Evaluation Needs Multiple Metrics

Different metrics measure different aspects of translation quality.

A system may have:

- high lexical overlap but poor semantics;
- good semantic adequacy but different wording;
- fluent output with an omitted source fact;
- correct Arabic words but incorrect tashkeel.

Therefore, no single metric should be treated as a complete measure of translation
quality.

In [ ]:
import math
import platform
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

metric_overview = pd.DataFrame(
    [
        ("BLEU", "word n-gram overlap", "corpus level"),
        ("chrF", "character n-gram overlap", "sentence/corpus"),
        ("COMET", "learned adequacy/quality estimate", "sentence/system"),
        ("Semantic similarity", "embedding-space similarity", "sentence/system"),
        ("Human evaluation", "adequacy/fluency/error judgments", "sentence/system"),
    ],
    columns=["Metric", "Main signal", "Typical use"],
)

metric_overview

# 2. Reference-Based Evaluation

Most MT metrics compare a candidate translation with one or more reference
translations.

Reference quality matters: a poor or narrow reference can penalize valid alternatives.

# 3. Corpus-Level vs Sentence-Level Scores

BLEU was designed primarily as a corpus-level metric.

Sentence-level variants can be useful for analysis, but they are less stable because
short sentences contain few higher-order n-grams.

# 4. BLEU

BLEU combines:

- modified n-gram precision;
- a geometric mean across n-gram orders;
- a brevity penalty.

Standard BLEU commonly uses n = 1 through 4.

# 5. Modified n-Gram Precision

Candidate n-gram counts are clipped by their maximum count in the reference.

This prevents a system from receiving unlimited credit for repeating the same word.

In [ ]:
def extract_ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


def clipped_ngram_counts(reference_tokens, candidate_tokens, n):
    reference_counts = Counter(
        extract_ngrams(reference_tokens, n)
    )
    candidate_counts = Counter(
        extract_ngrams(candidate_tokens, n)
    )

    clipped = sum(
        min(count, reference_counts[gram])
        for gram, count in candidate_counts.items()
    )

    total = sum(
        candidate_counts.values()
    )

    return clipped, total


reference = "the cat is on the mat".split()
candidate = "the cat the cat on mat".split()

clipped_ngram_counts(
    reference,
    candidate,
    n=1,
)

# 6. Brevity Penalty

BLEU penalizes candidates that are too short.

The corpus brevity penalty is:

```text
BP = 1                              if candidate_length > reference_length
BP = exp(1 - reference/reference_candidate_ratio) otherwise
```

In [ ]:
def brevity_penalty(reference_length, candidate_length):
    if candidate_length == 0:
        return 0.0

    if candidate_length > reference_length:
        return 1.0

    return math.exp(
        1.0
        - reference_length
        / candidate_length
    )


pd.Series({
    "equal length": brevity_penalty(10, 10),
    "short candidate": brevity_penalty(10, 8),
    "long candidate": brevity_penalty(10, 12),
})

# 7. Corpus BLEU Implementation

For numerical stability on a small teaching corpus, this implementation uses add-one
smoothing for n-gram orders above unigram.

In [ ]:
def corpus_bleu(
    references,
    hypotheses,
    max_order=4,
    smooth=True,
):
    clipped_totals = np.zeros(
        max_order,
        dtype=float,
    )

    candidate_totals = np.zeros(
        max_order,
        dtype=float,
    )

    total_reference_length = 0
    total_candidate_length = 0

    for reference, hypothesis in zip(
        references,
        hypotheses,
    ):
        ref_tokens = reference.split()
        hyp_tokens = hypothesis.split()

        total_reference_length += len(
            ref_tokens
        )

        total_candidate_length += len(
            hyp_tokens
        )

        for order in range(
            1,
            max_order + 1,
        ):
            clipped, total = (
                clipped_ngram_counts(
                    ref_tokens,
                    hyp_tokens,
                    order,
                )
            )

            clipped_totals[
                order - 1
            ] += clipped

            candidate_totals[
                order - 1
            ] += total

    precisions = []

    for order in range(
        1,
        max_order + 1,
    ):
        numerator = clipped_totals[
            order - 1
        ]

        denominator = candidate_totals[
            order - 1
        ]

        if denominator == 0:
            precisions.append(
                0.0
            )
            continue

        if (
            smooth
            and order > 1
        ):
            numerator += 1.0
            denominator += 1.0

        precisions.append(
            numerator
            / denominator
        )

    if any(
        precision <= 0
        for precision in precisions
    ):
        geometric_mean = 0.0
    else:
        geometric_mean = math.exp(
            sum(
                math.log(
                    precision
                )
                for precision
                in precisions
            )
            / max_order
        )

    bp = brevity_penalty(
        total_reference_length,
        total_candidate_length,
    )

    return 100.0 * bp * geometric_mean

# 8. BLEU Limitations

BLEU may under-reward:

- synonyms;
- paraphrases;
- legitimate word-order variation.

It may also hide sentence-level failures when the corpus average is acceptable.

# 9. chrF

chrF computes character n-gram precision and recall.

This is especially useful for morphologically rich languages because character overlap
captures partial morphological similarity.

# 10. chrF++

chrF++ extends chrF by incorporating word n-grams in addition to character n-grams.

The official SacreBLEU implementation exposes this through non-zero word order.

# 11. chrF Implementation

The following educational implementation computes character n-gram F-score with
beta = 2, giving recall more weight.

In [ ]:
def character_ngrams(text, n):
    characters = [
        character
        for character in text
        if not character.isspace()
    ]

    return [
        tuple(
            characters[i:i+n]
        )
        for i in range(
            len(characters)
            - n
            + 1
        )
    ]


def chrf_sentence(
    reference,
    hypothesis,
    char_order=6,
    beta=2.0,
):
    precisions = []
    recalls = []

    for n in range(
        1,
        char_order + 1,
    ):
        ref_counts = Counter(
            character_ngrams(
                reference,
                n,
            )
        )

        hyp_counts = Counter(
            character_ngrams(
                hypothesis,
                n,
            )
        )

        overlap = sum(
            (
                ref_counts
                & hyp_counts
            ).values()
        )

        hyp_total = sum(
            hyp_counts.values()
        )

        ref_total = sum(
            ref_counts.values()
        )

        if (
            hyp_total == 0
            or ref_total == 0
        ):
            continue

        precisions.append(
            overlap
            / hyp_total
        )

        recalls.append(
            overlap
            / ref_total
        )

    if (
        not precisions
        or not recalls
    ):
        return 0.0

    precision = float(
        np.mean(
            precisions
        )
    )

    recall = float(
        np.mean(
            recalls
        )
    )

    beta_squared = (
        beta ** 2
    )

    denominator = (
        beta_squared
        * precision
        + recall
    )

    if denominator == 0:
        return 0.0

    score = (
        (1 + beta_squared)
        * precision
        * recall
        / denominator
    )

    return 100.0 * score


def corpus_chrf(
    references,
    hypotheses,
):
    scores = [
        chrf_sentence(
            reference,
            hypothesis,
        )
        for reference, hypothesis
        in zip(
            references,
            hypotheses,
        )
    ]

    return float(
        np.mean(
            scores
        )
    )

# 12. Semantic Similarity

Embedding-based metrics compare sentence representations in a semantic vector space.

They can reward meaning-preserving paraphrases that have lower lexical overlap.

# 13. Offline Semantic Proxy

To keep the lesson offline, TF-IDF cosine similarity is used as a transparent proxy.

This is **not SBERT**. It only demonstrates the evaluation workflow.

In [ ]:
def tfidf_semantic_similarity(
    references,
    hypotheses,
):
    combined = list(
        references
    ) + list(
        hypotheses
    )

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        lowercase=True,
    )

    matrix = vectorizer.fit_transform(
        combined
    )

    n = len(
        references
    )

    ref_matrix = matrix[
        :n
    ]

    hyp_matrix = matrix[
        n:
    ]

    scores = []

    for index in range(n):
        score = cosine_similarity(
            ref_matrix[index],
            hyp_matrix[index],
        )[0, 0]

        scores.append(
            float(
                score
            )
        )

    return np.array(
        scores
    )

# 14. COMET

COMET is a learned MT evaluation framework.

Reference-based COMET models commonly use:

- source sentence;
- machine translation;
- reference translation.

The model produces sentence-level scores and a system-level score.

# 15. Optional COMET Workflow

This cell is disabled because COMET models are downloaded externally.

In [ ]:
RUN_COMET = False

comet_example = '''
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

data = [
    {
        "src": source,
        "mt": hypothesis,
        "ref": reference,
    }
    for source, hypothesis, reference
    in zip(sources, hypotheses, references)
]

output = model.predict(
    data,
    batch_size=8,
    gpus=0,
)

sentence_scores = output.scores
system_score = output.system_score
'''

print(comet_example)

# 16. Optional SentenceTransformer Workflow

This optional workflow computes real sentence embeddings when a suitable model is
available locally or can be downloaded.

In [ ]:
RUN_SBERT = False

sbert_example = '''
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

reference_embeddings = model.encode(
    references,
    normalize_embeddings=True,
)

hypothesis_embeddings = model.encode(
    hypotheses,
    normalize_embeddings=True,
)

sentence_scores = [
    float(
        cosine_similarity(
            reference_embeddings[i:i+1],
            hypothesis_embeddings[i:i+1],
        )[0, 0]
    )
    for i in range(len(references))
]
'''

print(sbert_example)

# 17. Evaluation Corpus

We create a small English target-side test set with source sentences for COMET-style
data organization.

In [ ]:
sources = [
    "أَنَا أَكْتُبُ الْكِتَابَ",
    "هُوَ يَكْتُبُ الْكِتَابَ",
    "هِيَ تَقْرَأُ الْكِتَابَ",
    "نَحْنُ نَقْرَأُ الْكِتَابَ",
    "ذَهَبْتُ إِلَى الْمَدْرَسَةِ",
    "هُوَ فِي الْمَدْرَسَةِ",
    "هِيَ فِي الْمَدْرَسَةِ",
    "كِتَابُهُ جَدِيدٌ",
    "كِتَابُهَا جَدِيدٌ",
    "وَسَيَكْتُبُونَهَا",
]

references = [
    "i write the book",
    "he writes the book",
    "she reads the book",
    "we read the book",
    "i went to the school",
    "he is in the school",
    "she is in the school",
    "his book is new",
    "her book is new",
    "and they will write it",
]

test_frame = pd.DataFrame({
    "source": sources,
    "reference": references,
})

test_frame

# 18. Competing MT Systems

System A is deliberately weaker than System B.

In [ ]:
system_a = [
    "i write book",
    "he write the book",
    "she read book",
    "we reads the book",
    "i go to school",
    "he in the school",
    "she is at school",
    "his book new",
    "her book is new",
    "they will write it",
]

system_b = [
    "i write the book",
    "he writes the book",
    "she reads the book",
    "we read the book",
    "i went to the school",
    "he is in the school",
    "she is in the school",
    "his book is new",
    "her book is new",
    "and they will write it",
]

systems = {
    "System A": system_a,
    "System B": system_b,
}

pd.DataFrame({
    "source": sources,
    "reference": references,
    "System A": system_a,
    "System B": system_b,
})

# 19. Corpus-Level Metrics

In [ ]:
corpus_metric_rows = []

for system_name, hypotheses in systems.items():
    corpus_metric_rows.append({
        "system": system_name,
        "BLEU": corpus_bleu(
            references,
            hypotheses,
        ),
        "chrF": corpus_chrf(
            references,
            hypotheses,
        ),
        "TFIDF_semantic": float(
            tfidf_semantic_similarity(
                references,
                hypotheses,
            ).mean()
            * 100
        ),
    })

corpus_metrics = pd.DataFrame(
    corpus_metric_rows
)

corpus_metrics

# 20. Sentence-Level Metrics

Sentence-level scores are essential for uncertainty estimation and error analysis.

In [ ]:
def sentence_bleu_like(
    reference,
    hypothesis,
):
    return corpus_bleu(
        [reference],
        [hypothesis],
        max_order=2,
        smooth=True,
    )


sentence_metric_rows = []

for system_name, hypotheses in systems.items():
    semantic_scores = (
        tfidf_semantic_similarity(
            references,
            hypotheses,
        )
        * 100
    )

    for index, (
        source,
        reference,
        hypothesis,
    ) in enumerate(
        zip(
            sources,
            references,
            hypotheses,
        )
    ):
        sentence_metric_rows.append({
            "system": system_name,
            "sentence_id": index,
            "source": source,
            "reference": reference,
            "hypothesis": hypothesis,
            "BLEU2_sentence": sentence_bleu_like(
                reference,
                hypothesis,
            ),
            "chrF": chrf_sentence(
                reference,
                hypothesis,
            ),
            "semantic_proxy": semantic_scores[
                index
            ],
        })

sentence_metrics = pd.DataFrame(
    sentence_metric_rows
)

sentence_metrics.head()

# 21. Mean

The mean summarizes central tendency but does not show variability.

# 22. Standard Deviation

Standard deviation describes dispersion among sentence-level scores.

A large standard deviation means translation quality varies substantially across
sentences.

In [ ]:
summary_rows = []

for system_name in systems:
    subset = sentence_metrics[
        sentence_metrics[
            "system"
        ]
        == system_name
    ]

    for metric in [
        "BLEU2_sentence",
        "chrF",
        "semantic_proxy",
    ]:
        values = subset[
            metric
        ].to_numpy()

        summary_rows.append({
            "system": system_name,
            "metric": metric,
            "mean": float(
                np.mean(values)
            ),
            "std": float(
                np.std(
                    values,
                    ddof=1,
                )
            ),
            "n": len(values),
        })

summary_frame = pd.DataFrame(
    summary_rows
)

summary_frame

# 23. Standard Error

The standard error estimates uncertainty in the sample mean:

```text
SE = SD / sqrt(n)
```

In [ ]:
summary_frame[
    "standard_error"
] = (
    summary_frame["std"]
    / np.sqrt(
        summary_frame["n"]
    )
)

summary_frame

# 24. 95% Confidence Interval

A simple normal-approximation interval is:

```text
mean ± 1.96 × standard_error
```

For small or irregular MT datasets, bootstrap intervals are often preferable.

In [ ]:
summary_frame[
    "CI95_lower_normal"
] = (
    summary_frame["mean"]
    - 1.96
    * summary_frame[
        "standard_error"
    ]
)

summary_frame[
    "CI95_upper_normal"
] = (
    summary_frame["mean"]
    + 1.96
    * summary_frame[
        "standard_error"
    ]
)

summary_frame

# 25. Bootstrap Confidence Interval

Bootstrap resampling repeatedly samples sentence pairs with replacement.

In [ ]:
def bootstrap_mean_ci(
    values,
    n_bootstrap=3000,
    confidence=0.95,
    seed=42,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    rng = np.random.default_rng(
        seed
    )

    n = len(values)

    bootstrap_means = np.empty(
        n_bootstrap,
        dtype=float,
    )

    for i in range(
        n_bootstrap
    ):
        sample = rng.choice(
            values,
            size=n,
            replace=True,
        )

        bootstrap_means[i] = (
            sample.mean()
        )

    alpha = (
        1.0
        - confidence
    )

    lower = np.quantile(
        bootstrap_means,
        alpha / 2,
    )

    upper = np.quantile(
        bootstrap_means,
        1 - alpha / 2,
    )

    return (
        float(lower),
        float(upper),
        bootstrap_means,
    )

In [ ]:
bootstrap_rows = []

for system_name in systems:
    subset = sentence_metrics[
        sentence_metrics[
            "system"
        ]
        == system_name
    ]

    for metric in [
        "BLEU2_sentence",
        "chrF",
        "semantic_proxy",
    ]:
        values = subset[
            metric
        ].to_numpy()

        lower, upper, _ = (
            bootstrap_mean_ci(
                values,
                n_bootstrap=3000,
                seed=SEED,
            )
        )

        bootstrap_rows.append({
            "system": system_name,
            "metric": metric,
            "mean": float(
                values.mean()
            ),
            "CI95_lower": lower,
            "CI95_upper": upper,
        })

bootstrap_summary = pd.DataFrame(
    bootstrap_rows
)

bootstrap_summary

# 26. Why Bootstrap MT Metrics

MT score distributions are often:

- non-normal;
- bounded;
- affected by sentence length;
- influenced by a few difficult examples.

Bootstrap resampling makes fewer distributional assumptions than a normal-theory CI.

# 27. Paired Bootstrap Comparison

Because both systems translate the **same sentences**, the comparison should be paired.

In [ ]:
def paired_bootstrap_difference(
    scores_a,
    scores_b,
    n_bootstrap=5000,
    seed=42,
):
    scores_a = np.asarray(
        scores_a,
        dtype=float,
    )

    scores_b = np.asarray(
        scores_b,
        dtype=float,
    )

    if len(scores_a) != len(
        scores_b
    ):
        raise ValueError(
            "Paired scores must have equal length."
        )

    rng = np.random.default_rng(
        seed
    )

    n = len(
        scores_a
    )

    differences = np.empty(
        n_bootstrap,
        dtype=float,
    )

    for i in range(
        n_bootstrap
    ):
        indices = rng.integers(
            0,
            n,
            size=n,
        )

        differences[i] = (
            scores_b[
                indices
            ].mean()
            - scores_a[
                indices
            ].mean()
        )

    observed = float(
        scores_b.mean()
        - scores_a.mean()
    )

    lower = float(
        np.quantile(
            differences,
            0.025,
        )
    )

    upper = float(
        np.quantile(
            differences,
            0.975,
        )
    )

    probability_b_better = float(
        np.mean(
            differences > 0
        )
    )

    return {
        "observed_difference_B_minus_A": observed,
        "CI95_lower": lower,
        "CI95_upper": upper,
        "P_bootstrap_B_better": probability_b_better,
    }


def get_scores(
    system_name,
    metric,
):
    return sentence_metrics[
        sentence_metrics[
            "system"
        ]
        == system_name
    ][
        metric
    ].to_numpy()


paired_bootstrap_difference(
    get_scores(
        "System A",
        "chrF",
    ),
    get_scores(
        "System B",
        "chrF",
    ),
)

# 28. Approximate Randomization

Approximate randomization tests the null hypothesis that the two systems are
exchangeable on each sentence.

For each iteration, sentence-level scores are randomly swapped between systems.

In [ ]:
def approximate_randomization_test(
    scores_a,
    scores_b,
    n_iterations=10000,
    seed=42,
):
    scores_a = np.asarray(
        scores_a,
        dtype=float,
    )

    scores_b = np.asarray(
        scores_b,
        dtype=float,
    )

    observed = abs(
        scores_b.mean()
        - scores_a.mean()
    )

    rng = np.random.default_rng(
        seed
    )

    extreme = 0

    for _ in range(
        n_iterations
    ):
        swap = rng.random(
            len(scores_a)
        ) < 0.5

        perm_a = np.where(
            swap,
            scores_b,
            scores_a,
        )

        perm_b = np.where(
            swap,
            scores_a,
            scores_b,
        )

        difference = abs(
            perm_b.mean()
            - perm_a.mean()
        )

        if difference >= observed:
            extreme += 1

    p_value = (
        extreme + 1
    ) / (
        n_iterations + 1
    )

    return float(
        p_value
    )


approximate_randomization_test(
    get_scores(
        "System A",
        "chrF",
    ),
    get_scores(
        "System B",
        "chrF",
    ),
)

# 29. Statistical Significance

A small p-value suggests that the observed difference is unlikely under the null
hypothesis of exchangeable system outputs.

A p-value does **not** tell us the size or practical importance of the improvement.

# 30. Practical Significance

A statistically detectable gain may still be too small to matter operationally.

Always report:

- score difference;
- confidence interval;
- significance test;
- practical interpretation.

# 31. Effect Size

For paired sentence-level scores, a standardized effect can be computed from the
difference vector.

In [ ]:
def paired_effect_size(
    scores_a,
    scores_b,
):
    differences = (
        np.asarray(
            scores_b,
            dtype=float,
        )
        - np.asarray(
            scores_a,
            dtype=float,
        )
    )

    sd = np.std(
        differences,
        ddof=1,
    )

    if sd == 0:
        return (
            float("inf")
            if differences.mean() != 0
            else 0.0
        )

    return float(
        differences.mean()
        / sd
    )


effect_sizes = pd.Series({
    metric: paired_effect_size(
        get_scores(
            "System A",
            metric,
        ),
        get_scores(
            "System B",
            metric,
        ),
    )
    for metric in [
        "BLEU2_sentence",
        "chrF",
        "semantic_proxy",
    ]
})

effect_sizes

# 32. Metric Correlation

Metrics can disagree because they emphasize different properties.

Correlation analysis helps reveal whether two metrics rank sentence-level outputs
similarly.

In [ ]:
system_a_scores = sentence_metrics[
    sentence_metrics[
        "system"
    ]
    == "System A"
][
    [
        "BLEU2_sentence",
        "chrF",
        "semantic_proxy",
    ]
]

system_a_scores.corr()

# 33. System Ranking Stability

Bootstrap distributions can estimate how often one system outranks another.

In [ ]:
ranking_stability = []

for metric in [
    "BLEU2_sentence",
    "chrF",
    "semantic_proxy",
]:
    result = paired_bootstrap_difference(
        get_scores(
            "System A",
            metric,
        ),
        get_scores(
            "System B",
            metric,
        ),
        n_bootstrap=5000,
        seed=SEED,
    )

    ranking_stability.append({
        "metric": metric,
        **result,
    })

ranking_stability_frame = pd.DataFrame(
    ranking_stability
)

ranking_stability_frame

# 34. Error Categories

Aggregate scores should be accompanied by interpretable translation-error categories.

In [ ]:
error_taxonomy = pd.DataFrame(
    [
        ("Lexical", "incorrect content word"),
        ("Morphology", "wrong inflection or derivation"),
        ("Agreement", "gender/number/person mismatch"),
        ("Omission", "source information missing"),
        ("Addition", "unsupported information added"),
        ("Word order", "incorrect ordering"),
        ("Named entity", "name translated/transliterated incorrectly"),
        ("Diacritization", "tashkeel missing or incorrect"),
    ],
    columns=[
        "Error category",
        "Description",
    ],
)

error_taxonomy

# 35. Per-Category Evaluation

A useful research report can stratify test sentences by linguistic difficulty.

In [ ]:
categories = [
    "simple",
    "agreement",
    "simple",
    "agreement",
    "tense",
    "copula",
    "copula",
    "possessive",
    "possessive",
    "complex_morphology",
]

category_frame = pd.DataFrame({
    "sentence_id": list(
        range(
            len(categories)
        )
    ),
    "category": categories,
})

category_scores = (
    sentence_metrics
    .merge(
        category_frame,
        on="sentence_id",
    )
    .groupby(
        [
            "system",
            "category",
        ],
        as_index=False,
    )[
        [
            "chrF",
            "semantic_proxy",
        ]
    ]
    .mean()
)

category_scores

# 36. Arabic-Specific Evaluation

Arabic evaluation should explicitly report whether the test set is:

- fully vocalized;
- partially vocalized;
- unvocalized.

Mixing these conditions makes metric interpretation difficult.

# 37. Tashkeel Preservation

When Arabic is the target language, a translation may be lexically correct but fail the
experiment if required tashkeel is missing.

In [ ]:
ARABIC_DIACRITICS = set(
    "\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652"
)

def tashkeel_sequence(text):
    return "".join(
        character
        for character in text
        if character in ARABIC_DIACRITICS
    )

def tashkeel_count(text):
    return len(
        tashkeel_sequence(
            text
        )
    )

vocalized_reference = (
    "هُوَ يَكْتُبُ الْكِتَابَ"
)

same_lexemes_without_tashkeel = (
    "هو يكتب الكتاب"
)

pd.Series({
    "Reference tashkeel count": (
        tashkeel_count(
            vocalized_reference
        )
    ),
    "Unvocalized count": (
        tashkeel_count(
            same_lexemes_without_tashkeel
        )
    ),
})

# 38. Tashkeel-Sensitive Exact Match

In [ ]:
def exact_match(
    reference,
    hypothesis,
):
    return float(
        reference
        == hypothesis
    )

pd.Series({
    "fully correct": exact_match(
        vocalized_reference,
        vocalized_reference,
    ),
    "diacritics removed": exact_match(
        vocalized_reference,
        same_lexemes_without_tashkeel,
    ),
})

# 39. Vocalized vs Unvocalized Evaluation

If normalization strips tashkeel before scoring, meaningful errors can disappear.

For a fully vocalized task, report both:

- surface-form metric;
- optional normalized diagnostic metric.

Do **not** replace the primary vocalized evaluation with an unvocalized one.

In [ ]:
def strip_tashkeel(text):
    return "".join(
        character
        for character in text
        if character
        not in ARABIC_DIACRITICS
    )

vocalization_comparison = pd.Series({
    "Surface chrF": chrf_sentence(
        vocalized_reference,
        same_lexemes_without_tashkeel,
    ),
    "Stripped diagnostic chrF": chrf_sentence(
        strip_tashkeel(
            vocalized_reference
        ),
        strip_tashkeel(
            same_lexemes_without_tashkeel
        ),
    ),
})

vocalization_comparison

# 40. Reporting BLEU Correctly

A research report should state:

- implementation;
- tokenization;
- case handling;
- smoothing;
- number of references;
- corpus/test set.

SacreBLEU is commonly used because its signature makes these settings reproducible.

# 41. Reporting chrF Correctly

Report at least:

- character n-gram order;
- beta;
- whether whitespace is included;
- whether word n-grams are enabled for chrF++.

# 42. Reporting COMET Correctly

Report:

- COMET model checkpoint;
- whether references are used;
- package/model version;
- sentence and system score aggregation;
- hardware/runtime configuration when relevant.

# 43. Reproducible Evaluation Table

In [ ]:
report_rows = []

for system_name in systems:
    sentence_subset = sentence_metrics[
        sentence_metrics[
            "system"
        ]
        == system_name
    ]

    chrf_values = sentence_subset[
        "chrF"
    ].to_numpy()

    semantic_values = sentence_subset[
        "semantic_proxy"
    ].to_numpy()

    chrf_lower, chrf_upper, _ = (
        bootstrap_mean_ci(
            chrf_values,
            seed=SEED,
        )
    )

    semantic_lower, semantic_upper, _ = (
        bootstrap_mean_ci(
            semantic_values,
            seed=SEED,
        )
    )

    corpus_row = corpus_metrics[
        corpus_metrics[
            "system"
        ]
        == system_name
    ].iloc[0]

    report_rows.append({
        "System": system_name,
        "BLEU": corpus_row[
            "BLEU"
        ],
        "chrF mean": float(
            chrf_values.mean()
        ),
        "chrF SD": float(
            chrf_values.std(
                ddof=1
            )
        ),
        "chrF 95% CI": (
            f"[{chrf_lower:.2f}, "
            f"{chrf_upper:.2f}]"
        ),
        "Semantic mean": float(
            semantic_values.mean()
        ),
        "Semantic 95% CI": (
            f"[{semantic_lower:.2f}, "
            f"{semantic_upper:.2f}]"
        ),
    })

report_table = pd.DataFrame(
    report_rows
)

report_table

# 44. Confidence Interval Visualization

In [ ]:
chrf_bootstrap = (
    bootstrap_summary[
        bootstrap_summary[
            "metric"
        ]
        == "chrF"
    ]
    .reset_index(
        drop=True
    )
)

means = chrf_bootstrap[
    "mean"
].to_numpy()

lower_error = (
    means
    - chrf_bootstrap[
        "CI95_lower"
    ].to_numpy()
)

upper_error = (
    chrf_bootstrap[
        "CI95_upper"
    ].to_numpy()
    - means
)

plt.figure(
    figsize=(7, 5)
)

plt.errorbar(
    chrf_bootstrap[
        "system"
    ],
    means,
    yerr=np.vstack(
        [
            lower_error,
            upper_error,
        ]
    ),
    fmt="o",
    capsize=5,
)

plt.ylabel(
    "Sentence-level chrF"
)

plt.title(
    "Bootstrap 95% Confidence Intervals"
)

plt.tight_layout()
plt.show()

# 45. Significance Matrix

In [ ]:
significance_rows = []

for metric in [
    "BLEU2_sentence",
    "chrF",
    "semantic_proxy",
]:
    scores_a = get_scores(
        "System A",
        metric,
    )

    scores_b = get_scores(
        "System B",
        metric,
    )

    bootstrap_result = (
        paired_bootstrap_difference(
            scores_a,
            scores_b,
            n_bootstrap=5000,
            seed=SEED,
        )
    )

    p_value = (
        approximate_randomization_test(
            scores_a,
            scores_b,
            n_iterations=10000,
            seed=SEED,
        )
    )

    significance_rows.append({
        "metric": metric,
        "difference_B_minus_A": (
            bootstrap_result[
                "observed_difference_B_minus_A"
            ]
        ),
        "CI95_lower": (
            bootstrap_result[
                "CI95_lower"
            ]
        ),
        "CI95_upper": (
            bootstrap_result[
                "CI95_upper"
            ]
        ),
        "p_approx_randomization": (
            p_value
        ),
        "paired_effect_size": (
            paired_effect_size(
                scores_a,
                scores_b,
            )
        ),
    })

significance_table = pd.DataFrame(
    significance_rows
)

significance_table

# 46. Interpretation

A strong conclusion should answer:

1. Which system scores higher?
2. How large is the difference?
3. How uncertain is the estimate?
4. Is the difference statistically detectable?
5. Is it practically meaningful?
6. Do multiple metrics agree?
7. Which linguistic categories remain weak?

# 47. Common Evaluation Mistakes

Avoid:

- reporting only BLEU;
- comparing BLEU from different tokenizers;
- computing confidence intervals from only corpus-level scores;
- using unpaired tests on paired translations;
- treating p < 0.05 as proof of large practical improvement;
- hiding Arabic tashkeel differences through normalization;
- reporting COMET without the model checkpoint;
- selecting the test set after seeing results.

# 48. Recommended MT Evaluation Protocol

For a serious experiment:

1. freeze the test set;
2. generate translations for all systems;
3. compute corpus BLEU;
4. compute chrF/chrF++;
5. compute COMET if available;
6. optionally compute semantic embedding similarity;
7. keep sentence-level scores for analysis;
8. report mean and standard deviation where appropriate;
9. compute bootstrap 95% confidence intervals;
10. perform a paired significance test;
11. report effect size;
12. analyze linguistic error categories;
13. report Arabic vocalization policy explicitly.

# 49. Reproducibility

In [ ]:
reproducibility = pd.Series(
    {
        "module": (
            "Module 9 • Machine Translation"
        ),
        "lesson": (
            "Lesson 55 • Machine Translation Evaluation"
        ),
        "sentences": len(
            references
        ),
        "systems": len(
            systems
        ),
        "bootstrap_samples": 5000,
        "randomization_iterations": 10000,
        "seed": SEED,
        "offline_core": True,
        "python": (
            platform.python_version()
        ),
    },
    name="Lesson 55 experiment",
)

reproducibility

# 50. Knowledge Check

1. Why is BLEU primarily a corpus-level metric?
2. What is modified n-gram precision?
3. Why does BLEU use a brevity penalty?
4. Why can chrF be useful for Arabic?
5. What does COMET use as input?
6. Why are semantic embeddings useful?
7. What does standard deviation describe?
8. What does a 95% confidence interval describe?
9. Why bootstrap MT scores?
10. Why must system comparisons be paired?
11. What does an approximate-randomization p-value test?
12. Why is effect size different from significance?
13. Why can metrics disagree?
14. Why should tashkeel not be stripped in a fully vocalized task?
15. What information should be included in a reproducible MT evaluation table?

# 51. Exercises

## Exercise 1
Add a third MT system.

## Exercise 2
Increase the test set to at least 100 sentence pairs.

## Exercise 3
Compute BLEU with SacreBLEU and compare it with the educational implementation.

## Exercise 4
Compute chrF++ with SacreBLEU.

## Exercise 5
Run COMET on the same systems.

## Exercise 6
Run a multilingual SentenceTransformer model and compute cosine similarity.

## Exercise 7
Compare normal-theory and bootstrap confidence intervals.

## Exercise 8
Perform paired significance tests for all metric pairs.

## Exercise 9
Create a fully vocalized Arabic target-side evaluation set.

## Exercise 10
Build a final publication-style table containing mean, SD, 95% CI, significance, and effect size.

## Challenge Exercises

1. Implement paired bootstrap directly on corpus BLEU by resampling sentence pairs and recomputing BLEU.
2. Compare approximate randomization with a paired permutation test.
3. Analyze metric correlation on at least 500 sentences.
4. Evaluate vocalized and unvocalized Arabic separately.
5. Reproduce a full MarianMT, M2M-100, NLLB, and mT5 comparison with BLEU, chrF, COMET, semantic similarity, SD, and confidence intervals.

# 52. Summary and Next Lesson

In this lesson:

- BLEU was implemented from modified n-gram precision and brevity penalty;
- chrF-style character evaluation was implemented;
- semantic-similarity evaluation was demonstrated with an offline TF-IDF proxy;
- current COMET and SentenceTransformer workflows were introduced as optional extensions;
- sentence-level means, standard deviations, standard errors, and 95% confidence
  intervals were calculated;
- bootstrap confidence intervals and paired bootstrap comparisons were implemented;
- approximate randomization and paired effect sizes were calculated;
- metric correlation, category-level evaluation, and ranking stability were analyzed;
- Arabic-specific evaluation, tashkeel preservation, and vocalized versus normalized
  diagnostics were treated explicitly;
- a reproducible publication-style evaluation table was produced.

## Next Lesson

**Lesson 56: Machine Translation End-to-End Project — Data Preparation, Fine-Tuning,
Evaluation, and Experiment Reporting** integrates dataset preparation, train/dev/test
splits, pretrained MT fine-tuning workflow design, decoding, BLEU, chrF, COMET,
semantic similarity, confidence intervals, significance testing, and final experiment
reporting.

# References

- Papineni, K. et al. *BLEU: a Method for Automatic Evaluation of Machine Translation*.
- Post, M. *A Call for Clarity in Reporting BLEU Scores*.
- Popović, M. work on chrF and character n-gram MT evaluation.
- Rei, R. et al. work on COMET.
- SacreBLEU official documentation and implementation.
- Unbabel COMET official documentation and repository.
- Sentence Transformers documentation.